# Mesma coisa do experiment_9_svr_riemann_features_std, porém usando features que foram selecionadas apenas com 80% do dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score

In [2]:
riemann_features = pd.read_csv("../results/feature_importance_mrmr_results.csv")["feature"].tolist()
random_forest_features = pd.read_csv("../results/random_forest_feature_selection_v4.csv")["feature"].tolist()
gevrey_features = pd.read_csv("../results/gevrey_method_feature_selection_v4.csv")["feature"].tolist()
correlation_features = pd.read_csv("../results/correlation_feature_selection_v4.csv")["feature"].tolist()


features_map = {
    "Random Forest (5)": random_forest_features[:5],
    "Random Forest (10)": random_forest_features[:10],
    f"Random Forest ({len(random_forest_features)})": random_forest_features,
    "Gevrey (10)": gevrey_features[:10],
    f"Gevrey ({len(gevrey_features)})": gevrey_features,
    "Correlation (10)": correlation_features[:10],
    "MRMR (10)": riemann_features[:10],
    "MRMR (50)": riemann_features[:50],
}


In [3]:
class DatasetService:
    LIMIT = 10_000

    def __init__(self, features: list[str]):
        dataset = pd.read_csv("../dataset/riemann_features.csv")
        self.X_df = dataset.drop(columns=["distance"])
        self.y_df = dataset["distance"]
        self.features = features

    def get_train_test(self):
        X = self.X_df[self.features].to_numpy()[:self.LIMIT]
        y = self.y_df.to_numpy()[:self.LIMIT]

        split = int(0.8 * len(X))
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]

        return X_train, X_test, y_train, y_test

In [4]:
def run_experiment(features_map):
    results = []

    param_grid = {
        "svr__C": [0.1, 1, 10, 100],
        "svr__epsilon": [0.001, 0.01, 0.1, 0.5],
        "svr__gamma": ["scale", 0.01, 0.1, 1.0]
    }

    for group_name, features in features_map.items():
        print(f"Running experiment for group: {group_name} with {len(features)} features")
        data = DatasetService(features)
        X_train, X_test, y_train, y_test = data.get_train_test()

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(kernel="rbf"))
        ])

        grid = GridSearchCV(
            pipeline,
            param_grid,
            scoring="neg_root_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=5),
            n_jobs=-1
        )

        grid.fit(X_train, y_train.ravel())

        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "Group": group_name,
            "RMSE": rmse,
            "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
            "Best gamma": grid.best_params_["svr__gamma"],
            "Features": len(features)
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


In [5]:
df_standard = run_experiment(features_map)
print(df_standard)
df_standard.to_csv("../results/experiment_9_svr_riemann_features_std_80_percent.csv", index=False)

Running experiment for group: Random Forest (5) with 5 features
Running experiment for group: Random Forest (10) with 10 features
Running experiment for group: Random Forest (20) with 20 features
Running experiment for group: Gevrey (10) with 10 features
Running experiment for group: Gevrey (24) with 24 features
Running experiment for group: Correlation (10) with 10 features
Running experiment for group: MRMR (10) with 10 features
Running experiment for group: MRMR (50) with 50 features
                Group      RMSE        R2  Best C  Best epsilon  Best gamma  \
0           MRMR (10)  0.026696  0.989133     100         0.001        0.10   
1  Random Forest (20)  0.040942  0.974440     100         0.010        0.01   
2         Gevrey (24)  0.041471  0.973775      10         0.001        0.01   
3           MRMR (50)  0.043937  0.970563      10         0.001        0.01   
4         Gevrey (10)  0.126746  0.755038     100         0.001        0.01   
5   Random Forest (5)  0.226165  0